# Phase 13 — FOLD 2 ONLY (Parallel Runner)
Run this alongside the main notebook. Trains fold 2 and saves to `achieved_phase13/`.

In [ ]:
FOLD_ID = 2  # << THIS NOTEBOOK TRAINS FOLD 2 ONLY

from google.colab import drive
import subprocess, sys, os, shutil, time, gc
drive.mount('/content/drive')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'h5py', 'scikit-learn'], timeout=120)
REPO_DIR = '/content/phase2'
if os.path.exists(REPO_DIR): shutil.rmtree(REPO_DIR)
subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/nithin12342/phase2.git', REPO_DIR], timeout=120, check=True)
PROJECT_ROOT = os.path.join(REPO_DIR, 'ml_pipeline', 'h5_omnifusion')
if PROJECT_ROOT not in sys.path: sys.path.insert(0, PROJECT_ROOT)

import torch, numpy as np, pandas as pd, glob
import torch.nn as nn
from torch.cuda.amp import autocast, GradScaler
from torch.optim import AdamW
from torch.optim.lr_scheduler import OneCycleLR
from sklearn.metrics import f1_score, roc_auc_score, accuracy_score, precision_score, recall_score, confusion_matrix
from tqdm.auto import tqdm

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3) if torch.cuda.is_available() else 0
AUTO_BS = 32 if vram_gb >= 14 else (16 if vram_gb >= 10 else 8)
ACCUM_STEPS = 4 if vram_gb >= 14 else (8 if vram_gb >= 10 else 16)
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
torch.backends.cudnn.benchmark = True
print(f'GPU: {torch.cuda.get_device_name(0)}, VRAM: {vram_gb:.1f}GB, BS={AUTO_BS}, Accum={ACCUM_STEPS}')

In [ ]:
# Data + Labels
root_dir = '/content/drive/MyDrive/DAIC-WOZ_Datasets'
H5_ROOT = os.path.join(root_dir, 'H5_OmniFusion_Output')
csv_files = glob.glob(os.path.join(H5_ROOT, '**', '*.csv'), recursive=True)
for extra in [os.path.join(root_dir, f) for f in ['all_labels.csv', 'merged_labels.csv', 'merged_all_labels.csv']]:
    if os.path.exists(extra) and extra not in csv_files: csv_files.append(extra)
all_dfs = []
for csv_path in csv_files:
    try:
        df = pd.read_csv(csv_path)
        id_col = next((c for c in ['Participant_ID','participant_id','ID','id','PID','filename'] if c in df.columns), None)
        phq_col = next((c for c in ['PHQ8_Score','phq8_score','PHQ_Score','phq_score','label','Label','depression'] if c in df.columns), None)
        if id_col and phq_col:
            m = df[[id_col, phq_col]].copy(); m.columns = ['Participant_ID', 'PHQ8_Score']
            m['Participant_ID'] = m['Participant_ID'].astype(str)
            if m['PHQ8_Score'].isin([0,1]).all() and m['PHQ8_Score'].nunique() <= 2:
                m['PHQ8_Score'] = m['PHQ8_Score'].map({1: 15, 0: 0})
            all_dfs.append(m)
    except: pass
merged_labels = pd.concat(all_dfs, ignore_index=True).drop_duplicates(subset='Participant_ID', keep='first')
MERGED_CSV = os.path.join(root_dir, 'phase13_labels.csv')
merged_labels.to_csv(MERGED_CSV, index=False)
SAVE_DIR = os.path.join(root_dir, 'checkpoints_phase13')
ACHIEVED_DIR = os.path.join(root_dir, 'achieved_phase13')
os.makedirs(SAVE_DIR, exist_ok=True); os.makedirs(ACHIEVED_DIR, exist_ok=True)
ALL_CKPT_DIRS = [os.path.join(root_dir, d) for d in ['checkpoints_phase13','checkpoints_phase12','checkpoints_phase11','checkpoints_phase10_finetune','h5_checkpoints']]
print(f'Labels: {len(merged_labels)}, Fold: {FOLD_ID}')

In [ ]:
# Train Fold
from src.models.h5_omnifusion import H5OmniFusion
from config.model_config import H5Config, ComputeTier
from src.data.h5_dataset import create_h5_dataloaders_kfold
from src.training.trainer import FocalLossBinary

def to_device(data, device):
    if isinstance(data, torch.Tensor): return data.to(device, non_blocking=True)
    if isinstance(data, dict): return {k: to_device(v, device) for k, v in data.items()}
    if isinstance(data, list): return [to_device(v, device) for v in data]
    return data

def find_best_checkpoint(fold, ckpt_dirs):
    for d in ckpt_dirs:
        if not os.path.isdir(d): continue
        for pat in [f'fold{fold}_phase13_best.pt',f'fold{fold}_phase12_best.pt',f'fold{fold}_phase12_latest.pt',f'fold{fold}_phase11_best.pt',f'h5_omnifusion_medium_fold{fold}_best.pt']:
            p = os.path.join(d, pat)
            if os.path.exists(p): return p
    for d in ckpt_dirs:
        if not os.path.isdir(d): continue
        for f in sorted(os.listdir(d)):
            if f.endswith('_best.pt'): return os.path.join(d, f)
    return None

fold = FOLD_ID
fold_csv = os.path.join(ACHIEVED_DIR, f'phase13_fold{fold}_preds.csv')
if os.path.exists(fold_csv):
    print(f'Fold {fold} already done! Results at {fold_csv}')
else:
    BATCH_SIZE=AUTO_BS; GRAD_ACCUM=ACCUM_STEPS; EFFECTIVE_BS=BATCH_SIZE*GRAD_ACCUM
    N_EPOCHS=15; PATIENCE=10; LR=3e-5*(EFFECTIVE_BS/32)
    FOCAL_ALPHA=0.50; FOCAL_GAMMA=2.0; LABEL_SMOOTHING=0.10
    LAMBDA_CLS=1.5; LAMBDA_PHQ=1.0; LAMBDA_ORTH=0.05; THRESHOLD=0.35

    train_loader, val_loader, test_loader = create_h5_dataloaders_kfold(
        h5_dir=H5_ROOT, labels_csv=MERGED_CSV, fold_idx=fold, n_folds=5, batch_size=BATCH_SIZE, seed=42, num_workers=4)
    print(f'Fold {fold}: Train={len(train_loader.dataset)}, Val={len(val_loader.dataset)}, Test={len(test_loader.dataset)}')

    model_config = H5Config.from_tier(ComputeTier.MEDIUM)
    model_config.loss.focal_alpha=FOCAL_ALPHA; model_config.loss.focal_gamma=FOCAL_GAMMA
    model_config.loss.label_smoothing=LABEL_SMOOTHING; model_config.loss.lambda_cls=LAMBDA_CLS
    model_config.loss.lambda_phq=LAMBDA_PHQ; model_config.loss.lambda_orth=LAMBDA_ORTH
    model_config.loss.decision_threshold=THRESHOLD; model_config.optimizer.lr=LR
    model_config.n_epochs=N_EPOCHS; model_config.patience=PATIENCE; model_config.mixed_precision=True
    model = H5OmniFusion(config=model_config)

    ckpt_path = find_best_checkpoint(fold, ALL_CKPT_DIRS)
    if ckpt_path:
        ckpt = torch.load(ckpt_path, map_location='cpu', weights_only=False)
        model.load_state_dict(ckpt.get('model_state_dict', ckpt.get('state_dict', ckpt)), strict=False)
        print(f'Loaded: {ckpt_path}')
    model = model.to(DEVICE)

    focal = FocalLossBinary(alpha=FOCAL_ALPHA, gamma=FOCAL_GAMMA, label_smoothing=LABEL_SMOOTHING)
    mse_loss = nn.MSELoss()
    optimizer = AdamW(model.parameters(), lr=LR, weight_decay=0.01)
    total_steps = (len(train_loader)//GRAD_ACCUM)*N_EPOCHS
    scheduler = OneCycleLR(optimizer, max_lr=LR, total_steps=max(total_steps,1), pct_start=0.1, anneal_strategy='cos', div_factor=25, final_div_factor=1000)
    scaler = GradScaler()
    best_j=-1.0; patience_ctr=0
    save_path=os.path.join(SAVE_DIR, f'fold{fold}_phase13_best.pt')
    latest_path=save_path.replace('_best.pt','_latest.pt')

    for epoch in range(N_EPOCHS):
        model.train(); train_loss=0.0; n_batches=0
        all_preds,all_labels_e,all_probs_e=[],[],[]
        optimizer.zero_grad()
        pbar = tqdm(train_loader, desc=f'Ep {epoch+1}/{N_EPOCHS}', leave=False)
        for step, batch in enumerate(pbar):
            batch = to_device(batch, DEVICE)
            try:
                with autocast():
                    outputs, orth_loss = model(batch)
                    logit=outputs['binary_logit'].squeeze(-1)
                    target_bin=batch['targets']['binary'].float()
                    loss_cls=focal(logit, target_bin)
                    loss_phq=mse_loss(outputs['phq_score'].squeeze(), batch['targets']['phq_score'].float())
                    loss_o=orth_loss if isinstance(orth_loss, torch.Tensor) else torch.tensor(0.0,device=DEVICE)
                    loss=(LAMBDA_CLS*loss_cls + LAMBDA_PHQ*loss_phq + LAMBDA_ORTH*loss_o)/GRAD_ACCUM
                scaler.scale(loss).backward()
                if (step+1)%GRAD_ACCUM==0 or (step+1)==len(train_loader):
                    scaler.unscale_(optimizer); torch.nn.utils.clip_grad_norm_(model.parameters(),0.5)
                    scaler.step(optimizer); scaler.update(); optimizer.zero_grad()
                    try: scheduler.step()
                    except: pass
                train_loss+=loss.item()*GRAD_ACCUM; n_batches+=1
                probs=outputs['binary_prob'].squeeze(-1).detach().float()
                all_preds.extend((probs>THRESHOLD).long().cpu().numpy())
                all_labels_e.extend(target_bin.cpu().numpy())
                all_probs_e.extend(probs.cpu().numpy())
                pbar.set_postfix({'loss':f'{loss.item()*GRAD_ACCUM:.4f}'})
            except RuntimeError as e:
                if 'out of memory' in str(e): torch.cuda.empty_cache(); optimizer.zero_grad(); continue
                raise

        def eval_amp(loader, desc=''):
            model.eval(); yt,yp,yprob,phqt,phqp=[],[],[],[],[]
            with torch.no_grad(), autocast():
                for b in tqdm(loader, desc=desc, leave=False):
                    bd=to_device(b,DEVICE); out,_=model(bd)
                    pr=out['binary_prob'].squeeze(-1).float().cpu().numpy(); lb=b['targets']['binary'].cpu().numpy()
                    yprob.extend(pr.flatten()); yt.extend(lb.flatten()); yp.extend((pr.flatten()>=THRESHOLD).astype(int))
                    phqt.extend(b['targets']['phq_score'].cpu().numpy().flatten())
                    pp=out['phq_score'].squeeze().float().cpu().numpy()
                    phqp.extend(pp.flatten() if np.ndim(pp)>0 else [float(pp)])
            yt=np.array(yt);yp=np.array(yp);yprob=np.array(yprob)
            cm=confusion_matrix(yt,yp,labels=[0,1]); tn,fp,fn,tp=cm.ravel()
            return {'f1':f1_score(yt,yp,zero_division=0),'auc':roc_auc_score(yt,yprob) if len(np.unique(yt))>1 else 0.5,
                    'recall':recall_score(yt,yp,zero_division=0),'spec':tn/(tn+fp) if (tn+fp)>0 else 0,
                    'tp':tp,'tn':tn,'fp':fp,'fn':fn,'y_true':yt,'y_prob':yprob,'y_pred':yp,
                    'phq_true':np.array(phqt),'phq_pred':np.array(phqp)}

        val_m=eval_amp(val_loader,'Val'); test_m=eval_amp(test_loader,'Test')
        val_j=val_m['recall']+val_m['spec']-1
        print(f'Ep {epoch+1}/{N_EPOCHS} | Loss={train_loss/max(n_batches,1):.4f} | '
              f'Val F1={val_m["f1"]:.3f} AUC={val_m["auc"]:.3f} Spec={val_m["spec"]:.3f} J={val_j:.3f} | '
              f'Test F1={test_m["f1"]:.3f} AUC={test_m["auc"]:.3f} Spec={test_m["spec"]:.3f}')
        print(f'  Val CM: TP={val_m["tp"]},TN={val_m["tn"]},FP={val_m["fp"]},FN={val_m["fn"]} | Test CM: TP={test_m["tp"]},TN={test_m["tn"]},FP={test_m["fp"]},FN={test_m["fn"]}')
        torch.save({'model_state_dict':model.state_dict(),'epoch':epoch,'val_j':val_j}, latest_path)
        if val_j > best_j:
            best_j=val_j; patience_ctr=0
            torch.save({'model_state_dict':model.state_dict(),'epoch':epoch,'val_j':val_j}, save_path)
            print(f'  NEW BEST J={val_j:.4f}')
        else: patience_ctr+=1
        if patience_ctr>=PATIENCE: print(f'  Early stopping'); break
        model.train()

    if os.path.exists(save_path):
        model.load_state_dict(torch.load(save_path, map_location=DEVICE, weights_only=False)['model_state_dict'])
    final=eval_amp(test_loader, f'Fold {fold} Final')
    pd.DataFrame({'y_true':final['y_true'],'y_prob':final['y_prob'],'y_pred':final['y_pred'],
                  'phq_true':final['phq_true'],'phq_pred':final['phq_pred']}).to_csv(fold_csv, index=False)
    print(f'\nFold {fold} COMPLETE! Best J={best_j:.4f}. Saved to {fold_csv}')
    del model; torch.cuda.empty_cache(); gc.collect()